In [4]:
import pandas as pd
import numpy as np  
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

Data Cleaning

In [5]:
df = pd.read_csv('/Users/aajinkyaingalkar/Machine Learning_Project/Customer_Churn/data/WA_Fn-UseC_-Telco-Customer-Churn.xls')
df.head(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [6]:
print(df.info(),df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:

df.drop(['customerID'], axis=1, inplace=True)

In [8]:
# df.info()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.isnull().sum()

gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [9]:
df[np.isnan(df['TotalCharges'])]

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


In [10]:
df[df['tenure'] == 0].index

Index([488, 753, 936, 1082, 1340, 3331, 3826, 4380, 5218, 6670, 6754], dtype='int64')

In [11]:
df.drop(labels=df[df['tenure'] == 0].index, axis=0, inplace=True)
df[df['tenure'] == 0].index

Index([], dtype='int64')

In [12]:
# Returns a DataFrame with only the rows where 'your_column' is NaN
df[df['TotalCharges'].isna() == True]


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [13]:
df.isnull().sum()
df.shape
df.fillna(df["TotalCharges"].mean())

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,No
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,No
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,No
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60,Yes


In [14]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [15]:
df["SeniorCitizen"]= df["SeniorCitizen"].map({0: "No", 1: "Yes"})
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,No,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [16]:
df["Churn"][df["Churn"]=="No"].groupby(by=df["gender"]).count()

gender
Female    2544
Male      2619
Name: Churn, dtype: int64

In [17]:
df["Churn"][df["Churn"]=="Yes"].groupby(by=df["gender"]).count()

gender
Female    939
Male      930
Name: Churn, dtype: int64

In [18]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   str    
 1   SeniorCitizen     7032 non-null   str    
 2   Partner           7032 non-null   str    
 3   Dependents        7032 non-null   str    
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   str    
 6   MultipleLines     7032 non-null   str    
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   str    
 9   OnlineBackup      7032 non-null   str    
 10  DeviceProtection  7032 non-null   str    
 11  TechSupport       7032 non-null   str    
 12  StreamingTV       7032 non-null   str    
 13  StreamingMovies   7032 non-null   str    
 14  Contract          7032 non-null   str    
 15  PaperlessBilling  7032 non-null   str    
 16  PaymentMethod     7032 non-null   str    
 17  MonthlyChar

In [19]:
df['gender'] = df['gender'].map({"Female" : 0, "Male" : 1})
df['SeniorCitizen'] = df['SeniorCitizen'].map({"No" : 0, "Yes" : 1})
df['Churn'] = df['Churn'].map({"No" : 0, "Yes" : 1})
df['Partner'] = df['Partner'].map({"No" : 0, "Yes" : 1})
df['Dependents'] = df['Dependents'].map({"No" : 0, "Yes" : 1})
df['PhoneService'] = df['PhoneService'].map({"No" : 0, "Yes" : 1})
df['PaperlessBilling'] = df['PaperlessBilling'].map({"No" : 0, "Yes" : 1})
df['MultipleLines'] = df['MultipleLines'].map({"No" : 0, "Yes" : 1, "No phone service" : 0})
df['OnlineBackup'] = df['OnlineBackup'].map({"No" : 0, "Yes" : 1, "No internet service" : 0})
df['OnlineSecurity'] = df['OnlineSecurity'].map({"No" : 0, "Yes" : 1, "No internet service" : 0})
df['DeviceProtection'] = df['DeviceProtection'].map({"No" : 0, "Yes" : 1, "No internet service" : 0})
df['TechSupport'] = df['TechSupport'].map({"No" : 0, "Yes" : 1, "No internet service" : 0})
df['StreamingTV'] = df['StreamingTV'].map({"No" : 0, "Yes" : 1, "No internet service" : 0})
df['StreamingMovies'] = df['StreamingMovies'].map({"No" : 0, "Yes" : 1, "No internet service" : 0})
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,1,0,1,0,0,DSL,0,1,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,1,0,0,0,34,1,0,DSL,1,0,1,0,0,0,One year,0,Mailed check,56.95,1889.50,0
2,1,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,1,0,0,0,45,0,0,DSL,1,0,1,1,0,0,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,0,0,0,0,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1


In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7032 non-null   int64  
 1   SeniorCitizen     7032 non-null   int64  
 2   Partner           7032 non-null   int64  
 3   Dependents        7032 non-null   int64  
 4   tenure            7032 non-null   int64  
 5   PhoneService      7032 non-null   int64  
 6   MultipleLines     7032 non-null   int64  
 7   InternetService   7032 non-null   str    
 8   OnlineSecurity    7032 non-null   int64  
 9   OnlineBackup      7032 non-null   int64  
 10  DeviceProtection  7032 non-null   int64  
 11  TechSupport       7032 non-null   int64  
 12  StreamingTV       7032 non-null   int64  
 13  StreamingMovies   7032 non-null   int64  
 14  Contract          7032 non-null   str    
 15  PaperlessBilling  7032 non-null   int64  
 16  PaymentMethod     7032 non-null   str    
 17  MonthlyChar

In [21]:
df = pd.get_dummies(df, columns=['Contract', 'PaymentMethod', 'InternetService'], drop_first=True)
df['StreamingMovies'].value_counts()

StreamingMovies
0    4301
1    2731
Name: count, dtype: int64

In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   int64  
 5   PhoneService                           7032 non-null   int64  
 6   MultipleLines                          7032 non-null   int64  
 7   OnlineSecurity                         7032 non-null   int64  
 8   OnlineBackup                           7032 non-null   int64  
 9   DeviceProtection                       7032 non-null   int64  
 10  TechSupport                            7032 non-null   int64  
 11  StreamingTV         

In [23]:
def object_to_int(dataframe_series):
    if dataframe_series.dtype=='object':
        dataframe_series = LabelEncoder().fit_transform(dataframe_series)
    return dataframe_series

In [24]:
df = df.apply(lambda x: object_to_int(x))
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No
0,0,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,False,False,False,True,False,False,False
1,1,0,0,0,34,1,0,1,0,1,...,56.95,1889.50,0,True,False,False,False,True,False,False
2,1,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,False,False,False,False,True,False,False
3,1,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,True,False,False,False,False,False,False
4,0,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,False,False,False,True,False,True,False


In [25]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   int64  
 5   PhoneService                           7032 non-null   int64  
 6   MultipleLines                          7032 non-null   int64  
 7   OnlineSecurity                         7032 non-null   int64  
 8   OnlineBackup                           7032 non-null   int64  
 9   DeviceProtection                       7032 non-null   int64  
 10  TechSupport                            7032 non-null   int64  
 11  StreamingTV         

In [26]:
boolean_columns = df.select_dtypes(include='bool').columns
boolean_columns
df[boolean_columns] = df[boolean_columns].astype(int)
df.head()   

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,MonthlyCharges,TotalCharges,Churn,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No
0,0,0,1,0,1,0,0,0,1,0,...,29.85,29.85,0,0,0,0,1,0,0,0
1,1,0,0,0,34,1,0,1,0,1,...,56.95,1889.50,0,1,0,0,0,1,0,0
2,1,0,0,0,2,1,0,1,1,0,...,53.85,108.15,1,0,0,0,0,1,0,0
3,1,0,0,0,45,0,0,1,0,1,...,42.30,1840.75,0,1,0,0,0,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,70.70,151.65,1,0,0,0,1,0,1,0


In [27]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7032 non-null   int64  
 1   SeniorCitizen                          7032 non-null   int64  
 2   Partner                                7032 non-null   int64  
 3   Dependents                             7032 non-null   int64  
 4   tenure                                 7032 non-null   int64  
 5   PhoneService                           7032 non-null   int64  
 6   MultipleLines                          7032 non-null   int64  
 7   OnlineSecurity                         7032 non-null   int64  
 8   OnlineBackup                           7032 non-null   int64  
 9   DeviceProtection                       7032 non-null   int64  
 10  TechSupport                            7032 non-null   int64  
 11  StreamingTV         

In [28]:
df.shape    

(7032, 24)

Data Imbalance

In [29]:
df['Churn'].value_counts()
# Clearly shows that the dataset is imbalanced.
#  The number of customers who have not churned is significantly higher than those who have churned. 
# This imbalance can affect the performance of machine learning models, as they may become biased towards the majority class.
#  To address this issue, techniques such as oversampling the minority class, undersampling the majority class, or using algorithms that are robust to class imbalance can be employed.

Churn
0    5163
1    1869
Name: count, dtype: int64

SMOTE (Synthetic Minority Over-sampling Technique)

How SMOTE Works
Instead of simply duplicating existing data points (which leads to overfitting), SMOTE synthesizes brand new points:
1. It looks at a minority class sample.
2. It finds its k-nearest neighbors (typically k=5) belonging to the same class.
3. It draws a line between the sample and a randomly selected neighbor.
4. It places a new synthetic data point somewhere along that line via interpolation.

In [30]:
X = df.drop(columns = ['Churn'])
y = df['Churn'].values

In [31]:
#train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((5625, 23), (1407, 23), (5625,), (1407,))

In [32]:
smote = SMOTE(random_state=42)
resampled = smote.fit_resample(X_train, y_train)
X_train_resampled = resampled[0]
y_train_resampled = resampled[1]
resampled_test = smote.fit_resample(X_test, y_test)
X_test_resampled = resampled_test[0]
y_test_resampled = resampled_test[1]
X_test_resampled.shape, y_test_resampled.shape

((2066, 23), (2066,))

In [33]:
print(pd.Series(y_train).value_counts())
print(pd.Series(y_train_resampled).value_counts())
X_train_resampled.shape[1]

0    4130
1    1495
Name: count, dtype: int64
1    4130
0    4130
Name: count, dtype: int64


23

 ========== ANN Used For Below With Accuray ======

In [34]:
#
model = tf.keras.Sequential([
    keras.layers.Dense(64, input_shape=(X_train_resampled.shape[1],), activation='relu'),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_resampled, y_train_resampled, epochs=100)

/Users/aajinkyaingalkar/Machine Learning_Project/Customer_Churn/.venv/lib/python3.14/site-packages/keras/src/layers/core/dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.6461 - loss: 2.3214
Epoch 2/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.7076 - loss: 1.5785
Epoch 3/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.7205 - loss: 1.4087
Epoch 4/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7303 - loss: 1.2491
Epoch 5/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7402 - loss: 1.1563
Epoch 6/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7482 - loss: 1.1730
Epoch 7/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7300 - loss: 1.3444
Epoch 8/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7660 - loss: 0.8489
Epoch 9/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7680 - loss: 0.8437
Epoch 10/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7604 - loss: 0.9863
Epoch 11/100
259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7626 - loss: 0.9457
Epoch 12/100
259/259 ━━━━━━━━━━━━━━━━━━

In [35]:
predictions = model.predict(X_test_resampled)
predictions_labels = np.argmax(predictions, axis=1)
print(predictions_labels[1010])
print(y_test_resampled[1010])
def get_accuracy(y_true, y_pred):
    accuracy = np.sum(y_true == y_pred) / len(y_true)
    return accuracy

accuracy = get_accuracy(y_test_resampled, predictions_labels)
print(f'Accuracy: {accuracy*100}%')
print(classification_report(y_test_resampled, predictions_labels))

65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
0
1
Accuracy: 50.0%
              precision    recall  f1-score   support

           0       0.50      1.00      0.67      1033
           1       0.00      0.00      0.00      1033

    accuracy                           0.50      2066
   macro avg       0.25      0.50      0.33      2066
weighted avg       0.25      0.50      0.33      2066



/Users/aajinkyaingalkar/Machine Learning_Project/Customer_Churn/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aajinkyaingalkar/Machine Learning_Project/Customer_Churn/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aajinkyaingalkar/Machine Learning_Project/Customer_Churn/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted sa

The Accuracy For ANN Is Not The Best

========================= Gotta Try It With Some Other Models =========================

In [37]:
# X_fit, X_val, y_fit, y_val = train_test_split(
#     X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
# )

# # Weight churners during training; don't also apply SMOTE here.
# negatives = (y_fit == 0).sum()
# positives = (y_fit == 1).sum()

# model = XGBClassifier(
#     eval_metric="logloss",
#     scale_pos_weight=negatives / positives,
#     random_state=42
# )
# model.fit(X_fit, y_fit)

# # Select a threshold using validation data
# val_scores = model.predict_proba(X_val)[:, 1]
# thresholds = np.arange(0.05, 0.96, 0.01)

# best_threshold = max(
#     thresholds,
#     key=lambda t: f1_score(y_val, (val_scores >= t).astype(int))
# )
# print("Best validation threshold for F1:", best_threshold)

# # Evaluate on the untouched test set
# test_scores = model.predict_proba(X_test)[:, 1]
# test_predictions = (test_scores >= best_threshold).astype(int)
# print(classification_report(y_test, test_predictions))

In [38]:
xgb_model = XGBClassifier()
xgb_model.fit(X_train_resampled, y_train_resampled)
xgb_predictions = xgb_model.predict(X_test_resampled)
# probabilities = xgb_model.predict_proba(X_test_resampled)[:, 1]  # Get probabilities for the positive class
# y_pred = (probabilities >= 0.25).astype(int)  # Convert probabilities to binary predictions
xgb_accuracy = get_accuracy(y_test_resampled, xgb_predictions)
print(f'XGBoost Accuracy: {xgb_accuracy*100:.2f}%')
print(classification_report(y_test_resampled, xgb_predictions))

XGBoost Accuracy: 81.32%
              precision    recall  f1-score   support

           0       0.81      0.82      0.81      1033
           1       0.82      0.81      0.81      1033

    accuracy                           0.81      2066
   macro avg       0.81      0.81      0.81      2066
weighted avg       0.81      0.81      0.81      2066



In [39]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
# churn_scores = xgb_model.predict_proba(X_test)[:, 1]
# print(f"ROC-AUC Score: {roc_auc_score(y_test, churn_scores):.2f}")
# print("Different Thresholds:")
# for threshold in [0.0009, 0.05, 0.55, 0.75, 0.9]:
#     y_pred = (churn_scores >= threshold).astype(int)
#     print(f"Threshold: {threshold}")
#     print(f"Recall: {recall_score(y_test, y_pred)*100:.2f}%")
#     print(f"Precision: {precision_score(y_test, y_pred)*100:.2f}%")
#     print(f"F1-Score: {f1_score(y_test, y_pred)*100:.2f}%")
#     print(f"ROC-AUC: {roc_auc_score(y_test, churn_scores)*100:.2f}%")
#     print(f'accuracy: {get_accuracy(y_test, y_pred)*100:.2f}%')
#     print("-" * 30)
churn_scores = xgb_model.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, churn_scores):.3f}")

for threshold in [0.0009, 0.05, 0.55, 0.75, 0.9]:
    y_pred = (churn_scores >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.2%}")
    print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.2%}")
    print(f"F1-score:  {f1_score(y_test, y_pred, zero_division=0):.2%}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.2%}")


ROC-AUC: 0.799

Threshold: 0.0009
Recall:    99.73%
Precision: 27.75%
F1-score:  43.42%
Accuracy:  30.92%

Threshold: 0.05
Recall:    90.11%
Precision: 38.91%
F1-score:  54.35%
Accuracy:  59.77%

Threshold: 0.55
Recall:    53.21%
Precision: 53.93%
F1-score:  53.57%
Accuracy:  75.48%

Threshold: 0.75
Recall:    37.43%
Precision: 64.81%
F1-score:  47.46%
Accuracy:  77.97%

Threshold: 0.9
Recall:    17.91%
Precision: 74.44%
F1-score:  28.88%
Accuracy:  76.55%


In [40]:
random_forest_model = RandomForestClassifier()
random_forest_model.fit(X_train_resampled, y_train_resampled)
rf_predictions = random_forest_model.predict(X_test_resampled)  
rf_accuracy = get_accuracy(y_test_resampled, rf_predictions)
print(f'Random Forest Accuracy: {rf_accuracy*100:.2f}%')

Random Forest Accuracy: 79.72%


==== We Saw That XGBoost Was Much Better In Initial Recall Therefore We Will Tune It With Optuna ====

In [42]:
import optuna
THRESHOLD = 0.3
def objective(trial):
    # Define the hyperparameter search space
    parameters = {
        "n_estimators": trial.suggest_int('n_estimators', 50, 300),
        "learning_rate": trial.suggest_float('learning_rate', 0.01, 0.3),
        "max_depth": trial.suggest_int('max_depth', 3, 20),
        "subsample": trial.suggest_float('subsample', 0.5, 1.0),
        "gamma": trial.suggest_float('gamma', 0.0, 5.0),
        "reg_alpha": trial.suggest_float('reg_alpha', 0.0, 5.0),
        "reg_lambda": trial.suggest_float('reg_lambda', 0.0, 5.0),
    }
    # Create and train the Random Forest model with the suggested hyperparameters
    model = XGBClassifier(**parameters)
    model.fit(X_train_resampled, y_train_resampled)
    proba = model.predict_proba(X_test_resampled)[:, 1]
    y_pred = (proba >= THRESHOLD).astype(int)
    return recall_score(y_test_resampled, y_pred)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)
print("Best hyperparameters:", study.best_params)
print("Best recall score:", study.best_value)


[I 2026-10-07 22:17:23,230] A new study created in memory with name: no-name-4b3da26a-7e08-448f-9129-e2d1e9f93985
[I 2026-10-07 22:17:24,419] Trial 0 finished with value: 0.9051306873184899 and parameters: {'n_estimators': 165, 'learning_rate': 0.06859574314662739, 'max_depth': 13, 'subsample': 0.6387148753986096, 'gamma': 3.9338284154118774, 'reg_alpha': 1.8737417911577974, 'reg_lambda': 3.8967947882219094}. Best is trial 0 with value: 0.9051306873184899.
[I 2026-10-07 22:17:25,095] Trial 1 finished with value: 0.9070667957405615 and parameters: {'n_estimators': 203, 'learning_rate': 0.17180082352948461, 'max_depth': 6, 'subsample': 0.5395018944114731, 'gamma': 1.4042448217079646, 'reg_alpha': 2.952106387192774, 'reg_lambda': 3.471445575047305}. Best is trial 1 with value: 0.9070667957405615.
[I 2026-10-07 22:17:25,508] Trial 2 finished with value: 0.914811229428848 and parameters: {'n_estimators': 76, 'learning_rate': 0.11205405274156213, 'max_depth': 11, 'subsample': 0.7341823508281

Best hyperparameters: {'n_estimators': 154, 'learning_rate': 0.16764564088835285, 'max_depth': 3, 'subsample': 0.9064275991444303, 'gamma': 4.870436683214136, 'reg_alpha': 2.7544580335522997, 'reg_lambda': 4.437112298600285}
Best recall score: 0.9341723136495643


In [46]:
best_params = study.best_params
best_model = XGBClassifier(**best_params)
best_model.fit(X_train_resampled, y_train_resampled)
best_proba = best_model.predict_proba(X_test_resampled)[:, 1]
best_y_pred = (best_proba >= THRESHOLD).astype(int)
print(f"Recall:    {recall_score(y_test_resampled, best_y_pred, zero_division=0):.2%}")
print(f"Precision: {precision_score(y_test_resampled, best_y_pred, zero_division=0):.2%}")
print(f"F1-score:  {f1_score(y_test_resampled, best_y_pred, zero_division=0):.2%}")
print(f"Accuracy:  {accuracy_score(y_test_resampled, best_y_pred):.2%}")

Recall:    93.42%
Precision: 73.05%
F1-score:  81.99%
Accuracy:  79.48%
